# GRU4Rec maximum sequence length experiment

Trains GRU4Rec at several **maximum history lengths** on the selected dataset, for seeds 42–46, using the team's shared data loader and original-style HR/NDCG evaluator. The length limits both what the model trains on and what it sees at test time, so this is a maximum-sequence-length experiment, not an evaluation-only truncation of one trained model.

GRU4Rec is our RecBole v1.2.0 cross-entropy adapter. It trains on the final target of each training history; it is not verified as the exact GRU4Rec configuration from the ADRec paper.

Before running: import this notebook into Kaggle; attach the private input containing `cmpe257_kaggle_code.zip` and the selected dataset's `dataset.pkl`; enable Internet for dependencies and a GPU for full training. No GitHub token or push is needed.

**Start with `RUN_MODE = 'smoke'`** (one length, one seed, tiny data). Its numbers only show that the pipeline runs. Then set `'full'`. Start a fresh kernel when changing the dataset or code bundle.

## 1. Load the shared code and dataset paths

Select `DATASET` (`'baby'`, `'beauty'` or `'ml-100k'`) in the first cell.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# Attach the PRIVATE Kaggle dataset containing these two files.
# Leave as None to discover a single matching file; set explicit paths if needed.
CODE_ZIP = None  # e.g. '/kaggle/input/cmpe257-training/cmpe257_kaggle_code.zip'
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATASET = 'ml-100k'  # 'baby', 'beauty', or 'ml-100k'
DATA_FILE = None  # e.g. '/kaggle/input/cmpe257-training/dataset.pkl'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
from src.data.catalog import find_dataset, dataset_spec
DATASET_PATH = find_dataset(INPUT_ROOT, DATASET, DATA_FILE)
ARCHIVE_ROOT = WORK_ROOT if DATASET == 'baby' else WORK_ROOT / 'exports' / DATASET
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, DATASET_PATH)

## 2. Install dependencies
If Kaggle requests a session restart after installation, restart and rerun from cell 1. Do not change package versions independently without recording the change.

In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)

In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Import every required model now, so dependency errors happen before training.
from sasrec import SASRec
from model import Att_Diffuse_model

## 3. Configure the experiment
`LENGTHS` and `SEEDS` must match what the other models use so the results can be compared. Outputs go to `history_length/gru4rec/<dataset>/`; smoke runs use separate `_smoke` folders and are never mixed with full results.

In [ ]:
import csv, hashlib, json, shutil, statistics

MODEL_KEY, MODEL_LABEL = 'gru4rec', 'GRU4Rec'
STAGES = ['gru4rec']

RUN_MODE = 'smoke'   # 'smoke' = tiny pipeline check (one seed, one length). Switch to 'full' after it passes.
LENGTHS = [10, 25]   # maximum history lengths to compare. 50 is the shared default used by the main runs.
SEEDS = [42, 43, 44, 45, 46]
RESUME_ROOT = None   # Folder with saved len<L>/... run folders from an earlier session (optional)

assert RUN_MODE in ('full', 'smoke')
if RUN_MODE == 'smoke':
    LENGTHS, SEEDS = [10], [42]
assert LENGTHS and len(LENGTHS) == len(set(LENGTHS)) and all(type(L) is int and L > 0 for L in LENGTHS)
assert SEEDS and len(SEEDS) == len(set(SEEDS)) and all(type(s) is int for s in SEEDS)
if RUN_MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Full training requires a GPU.')
if RESUME_ROOT is not None and not Path(RESUME_ROOT).is_dir():
    raise FileNotFoundError(RESUME_ROOT)

PREFIX = 'history_length' if RUN_MODE == 'full' else 'history_length_smoke'
RUNS_ROOT = WORK_ROOT / ('runs' if RUN_MODE == 'full' else 'smoke_runs') / DATASET
OUT = WORK_ROOT / PREFIX / MODEL_KEY / DATASET                 # small result files
MODELS = WORK_ROOT / (PREFIX + '_models') / MODEL_KEY / DATASET  # model weights (keep out of Git)
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

METRICS = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']
METADATA = ['config.json', 'results.json', 'history.json', 'environment.json', 'requirements-lock.txt']

cfg = make_config(MODEL_KEY, RUN_MODE, dataset=DATASET)
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print('Mode:', RUN_MODE, '| lengths:', LENGTHS, '| seeds:', SEEDS)
print('Results directory:', OUT)
print(dataset_summary(data))
print(json.dumps(cfg, indent=2))

## 4. Define the training and export helpers
These functions do not train anything by themselves. `run_stage` reuses a finished stage, resumes an interrupted one, or trains a new one. `export_length` checks every seed, copies the small result files, records checkpoint hashes, and writes the CSVs for one length.

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def seed_label(seeds):
    seeds = list(seeds)
    return (f'{seeds[0]}-{seeds[-1]}' if len(seeds) > 1 and seeds == list(range(seeds[0], seeds[-1] + 1))
            else ','.join(map(str, seeds)))


def write_csv(path, values):
    with Path(path).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(values[0]))
        writer.writeheader()
        writer.writerows(values)


def run_stage(stage_dir, stage_cfg, *, needs_test, pretrained_checkpoint=None):
    """Train one stage. Reuse it if finished, or resume it, when saved in the same folder."""
    stage_dir = Path(stage_dir)
    if not (stage_dir / 'latest.pt').exists() and RESUME_ROOT:
        previous = Path(RESUME_ROOT) / stage_dir.relative_to(RUNS_ROOT)
        if (previous / 'latest.pt').exists():
            shutil.copytree(previous, stage_dir, dirs_exist_ok=True)
    resume = None
    if (stage_dir / 'latest.pt').exists():
        saved_cfg = json.loads((stage_dir / 'config.json').read_text())
        if saved_cfg != stage_cfg:
            raise ValueError(f'Saved stage used different settings: {stage_dir}')
        report_path = stage_dir / 'results.json'
        report = json.loads(report_path.read_text()) if report_path.exists() else None
        finished = bool(report) and (report['early_stopped'] or report['epochs_completed'] >= stage_cfg['epochs'])
        if finished and (not needs_test or report.get('test')) and (stage_dir / 'best.pt').exists():
            print('Reusing completed stage:', stage_dir.parent.name, stage_dir.name)
            return
        resume = stage_dir / 'latest.pt'
    print('Training', stage_dir.parent.name, stage_dir.name, '(resuming)' if resume else '', flush=True)
    train(stage_cfg, DATASET_PATH, stage_dir, pretrained_checkpoint=pretrained_checkpoint,
          resume=resume, finalize=needs_test)


def run_dir_for(L, seed):
    return RUNS_ROOT / f'len{L}' / f'{MODEL_KEY}_{RUN_MODE}_len{L}_seed{seed}'


def export_length(L):
    """Validate every seed at this length, copy the small files, write this length's CSVs."""
    out = OUT / f'len{L}'
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for s in SEEDS:
        run_dir = run_dir_for(L, s)
        folder = run_dir / MODEL_KEY
        config = json.loads((folder / 'config.json').read_text())
        report = json.loads((folder / 'results.json').read_text())
        assert config['max_len'] == L and config['seed'] == s and config['mode'] == RUN_MODE, folder
        assert config.get('dataset', 'baby') == DATASET, folder
        assert report['test'] and report['test']['protocol'] == 'adrec_upstream', folder
        assert report['test']['units'] == 'percent', folder
        if MODEL_KEY == 'adrec':
            pre_config = json.loads((run_dir / 'pretrain' / 'config.json').read_text())
            assert pre_config == dict(config, model='pretrain'), 'Pretraining must match ADRec settings'
        hashes = {}
        for stage in STAGES:
            for name in METADATA:
                shutil.copyfile(run_dir / stage / name, out / f'{stage}_len{L}_seed{s}_{name}')
            for name in ('best.pt', 'latest.pt'):
                hashes[f'{stage}/{name}'] = file_hash(run_dir / stage / name)
        (out / f'{MODEL_KEY}_len{L}_seed{s}_checkpoints.json').write_text(json.dumps(
            {'dataset': DATASET, 'max_len': L, 'seed': s, 'location': str(run_dir), 'files': hashes}, indent=2))
        shutil.make_archive(str(MODELS / f'{MODEL_KEY}_len{L}_seed{s}'), 'zip', root_dir=run_dir)
        rows.append(dict(model=MODEL_LABEL, dataset=DATASET, max_len=L, seed=s,
                         best_epoch=report['best_epoch'], epochs_completed=report['epochs_completed'],
                         **{k: report['test']['metrics'][k] for k in METRICS}))
    common = dict(model=MODEL_LABEL, dataset=DATASET, max_len=L, n_runs=len(rows), seeds=seed_label(SEEDS))
    summary = [dict(common, statistic='mean',
                    **{k: round(statistics.mean(r[k] for r in rows), 4) for k in METRICS})]
    if len(rows) > 1:
        summary.append(dict(common, statistic='std (sample, n-1)',
                            **{k: round(statistics.stdev(r[k] for r in rows), 4) for k in METRICS}))
    write_csv(out / f'{MODEL_KEY}_{DATASET}_len{L}_runs.csv', rows)
    write_csv(out / f'{MODEL_KEY}_{DATASET}_len{L}_summary.csv', summary)
    return rows, summary

## 5. Train every length and seed
For each length and seed, the shared `train()` function trains GRU4Rec, selects the checkpoint by validation, and evaluates once on the test set. After each length the results are exported. Rerunning the cell reuses finished runs.

In [ ]:
for L in LENGTHS:
    for s in SEEDS:
        run_cfg = make_config('gru4rec', RUN_MODE, dataset=DATASET)
        run_cfg['seed'] = s
        run_cfg['max_len'] = L
        print('max_len', L, 'seed', s, flush=True)
        run_stage(run_dir_for(L, s) / 'gru4rec', run_cfg, needs_test=True)
    export_length(L)
    print('Finished max_len', L, flush=True)

## 6. Combine the lengths and print the results
This cell reads the per-length CSVs from disk, so it also works after a resumed session. Values are percentages (mean and sample standard deviation over seeds).

In [ ]:
all_rows, all_summary = [], []
for L in LENGTHS:
    for name, target in ((f'{MODEL_KEY}_{DATASET}_len{L}_runs.csv', all_rows),
                         (f'{MODEL_KEY}_{DATASET}_len{L}_summary.csv', all_summary)):
        with (OUT / f'len{L}' / name).open() as handle:
            target += list(csv.DictReader(handle))
write_csv(OUT / f'{MODEL_KEY}_{DATASET}_history_length_runs.csv', all_rows)
write_csv(OUT / f'{MODEL_KEY}_{DATASET}_history_length_summary.csv', all_summary)

(OUT / 'EXPERIMENT.md').write_text(f"""# Experiment record: {MODEL_LABEL} maximum sequence length on {DATASET}

- Model: {MODEL_LABEL}, {RUN_MODE} mode. Maximum history lengths: {LENGTHS}. Seeds: {seed_label(SEEDS)}.
- What changes: the maximum sequence length, in BOTH training and testing{" (embedding pretraining and ADRec are both retrained at each length)" if MODEL_KEY == 'adrec' else ""}. This is not an evaluation-only truncation of a trained model.
- Dataset: `{DATASET}`, SHA-256 `{cfg['dataset_sha256']}`. Released train/validation/test splits retained.
- Everything else matches the shared settings (hidden size 128, batch size 512, up to 500 epochs, validation every 5 epochs, patience 4).
- Selection: improvement in any tracked validation metric (the shared runner's rule). Test is evaluated once, at the end.
- Evaluation: `adrec_upstream`, HR/NDCG percentages, no seen-item filtering. Mean and sample standard deviation (n-1) over seeds.
- Notebook: this file. Weights are stored separately from these result files.
""")

archive = shutil.make_archive(str(WORK_ROOT / f'{MODEL_KEY}_history_length_results'
                                  f"{'' if RUN_MODE == 'full' else '_smoke'}"), 'zip', root_dir=OUT)
for row in all_summary:
    print('max_len', row['max_len'], '|', row['statistic'], '| HR@20', row['HR@20'], '| NDCG@20', row['NDCG@20'])
print('Results zip:', archive)
print('Model files folder:', MODELS)

## 7. Download the output files
Before ending the session, download from Kaggle's Output panel:

- `gru4rec_history_length_results.zip`: the small CSV and metadata files, including per-length folders (`len10/`, `len25/`).
- `history_length_models/gru4rec/<dataset>/`: one model ZIP per length and seed. Keep weights outside Git.

**Save & Run All** starts a separate execution from the first cell. To continue an interrupted full run in a new session, attach the saved `runs/<dataset>/` folder and set `RESUME_ROOT` to it before running the setup again.